In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
RADIOMICS-DRIVEN DIGITAL TWIN + INTERACTIVE 3D VISUALIZATION
================================================================================
Pipeline kết hợp:
1. Batch Simulation & Cox Survival Analysis with Stratified K-Fold CV
2. Print α, β parameters for ALL patients
3. Case Study: Tìm 4 bệnh nhân đặc trưng (2 High Risk + 2 Low Risk)
4. Interactive HTML: Animation với Play/Stop + Opacity Slider

Author: Paul Nguyen
Modified: 4 representative patients (2 high + 2 low risk)
"""

import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from scipy.ndimage import zoom, binary_erosion, binary_dilation
from scipy.ndimage import label as ndimage_label
from scipy import ndimage
from skimage import measure
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# 1. CẤU HÌNH HỆ THỐNG
# ============================================================================

LUNG1_ROOT = Path("./nsclc/manifest-1603198545583/NSCLC-Radiomics")
CLINICAL_FILE = Path("./nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv")
OUTPUT_DIR = Path("./digital_twin_output")

N_JOBS = -1
DEBUG_MODE = False

# Tham số cho Batch Simulation (nhanh)
MLPA_CONFIG_BATCH = {
    'grid_size': 90,
    'num_iterations': 120,
    'mutation_chance': 0.005,
}

# Tham số cho 3D Visualization (chi tiết hơn)
MLPA_CONFIG_VIZ = {
    'grid_size': 90,
    'num_iterations': 120,
    'mutation_chance': 0.002,
    'necrosis_delay': 15,
    'capture_interval': 3,
}


# ============================================================================
# 2. DATA LOADING & RADIOMICS ENGINE
# ============================================================================

def load_patient_data_smart(patient_id: str, data_root: Path) -> Optional[Tuple]:
    """Load CT và Segmentation từ DICOM."""
    p_dir = data_root / patient_id
    if not p_dir.exists():
        return None

    ct_dirs = []
    seg_file = None
    
    for study in p_dir.glob("*"):
        if not study.is_dir():
            continue
        for subdir in study.iterdir():
            if subdir.is_dir():
                if subdir.name.startswith("300."):
                    segs = list(subdir.glob("*.dcm"))
                    seg_file = segs[0] if segs else None
                else:
                    dcm_count = len(list(subdir.glob("*.dcm")))
                    if dcm_count > 10:
                        ct_dirs.append((subdir, dcm_count))

    if not ct_dirs or not seg_file:
        return None
    
    ct_dir = max(ct_dirs, key=lambda x: x[1])[0]

    try:
        slices = [pydicom.dcmread(str(f)) for f in sorted(ct_dir.glob("*.dcm"))]
        slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
        volume = np.stack([s.pixel_array for s in slices])
        slope = float(getattr(slices[0], 'RescaleSlope', 1))
        intercept = float(getattr(slices[0], 'RescaleIntercept', 0))
        volume = volume * slope + intercept

        pixel_spacing = [float(x) for x in slices[0].PixelSpacing]
        z_positions = [float(s.ImagePositionPatient[2]) for s in slices]
        z_spacing = abs(z_positions[1] - z_positions[0]) if len(z_positions) > 1 else 1.0
        spacing = np.array([z_spacing, pixel_spacing[0], pixel_spacing[1]])
        origin = np.array([
            float(slices[0].ImagePositionPatient[2]),
            float(slices[0].ImagePositionPatient[1]),
            float(slices[0].ImagePositionPatient[0])
        ])

        seg_ds = pydicom.dcmread(str(seg_file))
        sop_to_z = {ds.SOPInstanceUID: idx for idx, ds in enumerate(slices)}

        tumor_mask = np.zeros_like(volume, dtype=np.uint8)
        lung_mask = np.zeros_like(volume, dtype=np.uint8)
        
        frames = seg_ds.pixel_array
        if frames.ndim == 2:
            frames = frames[np.newaxis, ...]

        seg_info = {}
        if hasattr(seg_ds, 'SegmentSequence'):
            for item in seg_ds.SegmentSequence:
                seg_info[int(item.SegmentNumber)] = getattr(item, 'SegmentLabel', '').lower()

        if hasattr(seg_ds, 'PerFrameFunctionalGroupsSequence'):
            for i, meta in enumerate(seg_ds.PerFrameFunctionalGroupsSequence):
                try:
                    seg_num = int(meta.SegmentIdentificationSequence[0].ReferencedSegmentNumber)
                    ref_uid = meta.DerivationImageSequence[0].SourceImageSequence[0].ReferencedSOPInstanceUID
                    if ref_uid in sop_to_z:
                        z = sop_to_z[ref_uid]
                        mask_slice = (frames[i] > 0).astype(np.uint8)
                        name = seg_info.get(seg_num, '')
                        
                        if any(k in name for k in ['gtv', 'tumor', 'neoplasm']):
                            tumor_mask[z] = np.maximum(tumor_mask[z], mask_slice)
                        elif any(k in name for k in ['lung', 'left', 'right']):
                            lung_mask[z] = np.maximum(lung_mask[z], mask_slice)
                except:
                    continue

        if np.sum(tumor_mask) == 0 or np.sum(lung_mask) == 0:
            return None

        for z in range(lung_mask.shape[0]):
            lung_mask[z] = ndimage.binary_fill_holes(lung_mask[z])

        lung_mask = np.logical_or(lung_mask, tumor_mask).astype(np.uint8)

        return volume, tumor_mask, lung_mask, spacing, origin

    except Exception:
        return None


def calculate_radiomics_phenotype(ct_volume: np.ndarray, tumor_mask: np.ndarray) -> Optional[Tuple]:
    """Tính toán Radiomics và map sang tham số sinh học."""
    voxels = ct_volume[tumor_mask > 0]
    if len(voxels) == 0:
        return None
    
    try:
        hist, _ = np.histogram(voxels, bins=64, density=True)
        hist = hist[hist > 0]
        entropy = -np.sum(hist * np.log2(hist))

        verts, faces, _, _ = measure.marching_cubes(tumor_mask, level=0.5)
        area = measure.mesh_surface_area(verts, faces)
        vol = np.sum(tumor_mask)
        sphericity = (np.pi**(1/3) * (6 * vol)**(2/3)) / area if area > 0 else 0
    except:
        return None

    entropy_normalized = np.clip((entropy - 0.2) / 0.6, 0, 1)
    
    p_alpha = 0.01 + (0.05 * entropy_normalized)
    p_necrosis = 0.01 + (0.08 * (1.0 - np.clip(sphericity, 0, 1)))

    return p_alpha, p_necrosis, entropy, sphericity


def run_simulation_batch(ct_volume, tumor_mask, lung_mask, config) -> Optional[Dict]:
    """Chạy simulation nhanh cho batch processing."""
    params = calculate_radiomics_phenotype(ct_volume, tumor_mask)
    if params is None:
        return None
    
    p_alpha, p_necrosis, p_entropy, p_sphericity = params

    scale = np.array([config['grid_size']] * 3) / np.array(tumor_mask.shape)
    tumor_grid = zoom(tumor_mask.astype(float), scale, order=0) > 0.5
    lung_grid = zoom(lung_mask.astype(float), scale, order=0) > 0.5
    tumor_grid = tumor_grid & lung_grid
    
    if np.sum(tumor_grid) == 0:
        return None

    ct_small = zoom(ct_volume, scale, order=1)
    microenv = np.clip((ct_small + 1000) / 1400, 0, 1)

    grid = np.zeros_like(tumor_grid, dtype=np.int8)
    grid[lung_grid] = 1
    grid[tumor_grid] = 2
    init_count = np.sum(grid == 2)

    for it in range(config['num_iterations']):
        cells = (grid == 2) | (grid == 4) | (grid == 6)
        bound = binary_dilation(cells) & (grid == 1)
        cands = np.argwhere(bound)
        
        if len(cands) > 0:
            n_grow = int(len(cands) * p_alpha)
            if n_grow > 0:
                n_grow = min(n_grow, len(cands))
                w = microenv[tuple(cands.T)]
                w_sum = w.sum()
                prob = w / w_sum if w_sum > 0 else None
                idx = np.random.choice(len(cands), n_grow, p=prob, replace=False)
                for i in idx:
                    grid[tuple(cands[i])] = 4

        mask_4 = grid == 4
        grid[mask_4 & (np.random.rand(*grid.shape) < 0.2)] = 2
        
        mask_2 = grid == 2
        grid[mask_2 & (np.random.rand(*grid.shape) < config['mutation_chance'])] = 6
        
        if it > 15:
            ero = binary_erosion(cells, iterations=2)
            grid[ero & (grid == 2) & (np.random.rand(*grid.shape) < p_necrosis)] = 3
        
        grid[~lung_grid] = 0

    total = np.sum((grid == 2) | (grid == 4) | (grid == 6))
    if total == 0:
        return None

    return {
        'Sim_Growth_Rate': (total - init_count) / config['num_iterations'],
        'Sim_Necrosis_Ratio': np.sum(grid == 3) / total,
        'Radiomics_Entropy': p_entropy,
        'Radiomics_Sphericity': p_sphericity,
        'Bio_Alpha': p_alpha,
        'Bio_Necrosis': p_necrosis,
    }


def process_wrapper(pid: str) -> Optional[Dict]:
    """Wrapper cho parallel processing."""
    try:
        data = load_patient_data_smart(pid, LUNG1_ROOT)
        if data is None:
            return None
        vol, tumor, lung, spacing, origin = data
        feats = run_simulation_batch(vol, tumor, lung, MLPA_CONFIG_BATCH)
        if feats:
            feats['PatientID'] = pid
        return feats
    except:
        return None


# ============================================================================
# 3. INTERACTIVE 3D VISUALIZATION ENGINE  
# ============================================================================

class RealSpaceTransform:
    """Chuyển đổi tọa độ grid sang không gian thực (mm)."""
    
    def __init__(self, spacing, origin, original_shape, grid_size):
        self.spacing = spacing
        self.origin = origin
        self.scale = np.array(original_shape) / grid_size

    def grid_to_mm(self, grid_coords):
        return (grid_coords * self.scale) * self.spacing + self.origin


class MLPA3DVisualizer:
    """Mô phỏng 3D với Interactive HTML."""

    def __init__(self, config: Dict):
        self.cfg = config
        self.grid_size = config['grid_size']
        self.transform = None
        self.left_lung_surface_mm = None
        self.right_lung_surface_mm = None
        self.plot_limits = None
        self.simulation_states = []

    def resample(self, mask, shape):
        scale = np.array([self.grid_size] * 3) / np.array(shape)
        return zoom(mask.astype(float), scale, order=0) > 0.5

    def separate_lungs(self, lung_grid):
        """Tách 2 lá phổi dựa trên connected components."""
        labeled, num_features = ndimage_label(lung_grid)
        
        if num_features < 2:
            mid_x = lung_grid.shape[2] // 2
            left_lung = lung_grid.copy()
            right_lung = lung_grid.copy()
            left_lung[:, :, mid_x:] = False
            right_lung[:, :, :mid_x] = False
            return left_lung, right_lung
        
        regions = []
        for i in range(1, num_features + 1):
            region_mask = (labeled == i)
            region_size = np.sum(region_mask)
            coords = np.argwhere(region_mask)
            centroid_x = coords[:, 2].mean() if len(coords) > 0 else 0
            regions.append((i, region_size, centroid_x, region_mask))
        
        regions.sort(key=lambda x: x[1], reverse=True)
        top_regions = regions[:2]
        
        if top_regions[0][2] < top_regions[1][2]:
            right_lung = top_regions[0][3]
            left_lung = top_regions[1][3]
        else:
            left_lung = top_regions[0][3]
            right_lung = top_regions[1][3]
        
        return left_lung, right_lung

    def prepare_lung_surfaces(self, lung_grid):
        """Trích xuất bề mặt 2 lá phổi riêng biệt."""
        print("   🔨 Đang tách và trích xuất bề mặt 2 lá phổi...")
        
        left_lung, right_lung = self.separate_lungs(lung_grid)
        
        def extract_surface(lung_part, name):
            eroded = binary_erosion(lung_part, iterations=1)
            surface_grid = lung_part & ~eroded
            coords = np.argwhere(surface_grid)
            
            if len(coords) > 0:
                mm_coords = self.transform.grid_to_mm(coords)
                step = max(1, len(mm_coords) // 1500)
                print(f"      - {name}: {len(coords)} voxels → {len(mm_coords[::step])} points")
                return mm_coords[::step]
            return None
        
        self.left_lung_surface_mm = extract_surface(left_lung, "Left Lung")
        self.right_lung_surface_mm = extract_surface(right_lung, "Right Lung")
        
        all_coords = np.argwhere(lung_grid)
        if len(all_coords) > 0:
            mm_coords = self.transform.grid_to_mm(all_coords)
            self.plot_limits = {
                'x': (mm_coords[:, 2].min(), mm_coords[:, 2].max()),
                'y': (mm_coords[:, 1].min(), mm_coords[:, 1].max()),
                'z': (mm_coords[:, 0].min(), mm_coords[:, 0].max())
            }

    def capture_state(self, grid, iteration):
        """Lưu trạng thái simulation để animate."""
        state_data = {'iteration': iteration, 'cells': {}}
        
        cell_configs = [
            (2, 'darkblue', 'Tumor Core', 4, 0.7),
            (4, 'red', 'Proliferating', 6, 1.0),
            (6, 'purple', 'Malignant', 5, 0.9),
            (3, 'dimgray', 'Necrotic', 4, 0.5),
        ]
        
        for state, color, name, size, alpha in cell_configs:
            coords = np.argwhere(grid == state)
            if len(coords) > 0:
                mm = self.transform.grid_to_mm(coords)
                step = max(1, len(mm) // 1000)
                mm = mm[::step]
                state_data['cells'][state] = {
                    'x': mm[:, 2].tolist(),
                    'y': mm[:, 1].tolist(),
                    'z': mm[:, 0].tolist(),
                    'color': color, 'name': name,
                    'size': size, 'alpha': alpha
                }
            else:
                state_data['cells'][state] = {
                    'x': [], 'y': [], 'z': [],
                    'color': color, 'name': name,
                    'size': size, 'alpha': alpha
                }
        
        self.simulation_states.append(state_data)

    def create_interactive_html(self, patient_id, risk_label, p_alpha, p_necrosis, output_path):
        """Tạo HTML interactive với Animation + Opacity Slider."""
        print(f"   🌐 Đang tạo HTML interactive...")
        
        traces = []
        
        if self.left_lung_surface_mm is not None:
            traces.append(go.Scatter3d(
                x=self.left_lung_surface_mm[:, 2],
                y=self.left_lung_surface_mm[:, 1],
                z=self.left_lung_surface_mm[:, 0],
                mode='markers',
                marker=dict(size=2, color='dodgerblue', opacity=0.15),
                name='Left Lung', hoverinfo='name'
            ))
        else:
            traces.append(go.Scatter3d(x=[], y=[], z=[], mode='markers', name='Left Lung'))
        
        if self.right_lung_surface_mm is not None:
            traces.append(go.Scatter3d(
                x=self.right_lung_surface_mm[:, 2],
                y=self.right_lung_surface_mm[:, 1],
                z=self.right_lung_surface_mm[:, 0],
                mode='markers',
                marker=dict(size=2, color='limegreen', opacity=0.15),
                name='Right Lung', hoverinfo='name'
            ))
        else:
            traces.append(go.Scatter3d(x=[], y=[], z=[], mode='markers', name='Right Lung'))
        
        cell_order = [2, 4, 6, 3]
        initial_state = self.simulation_states[0]
        
        for state in cell_order:
            cell_data = initial_state['cells'][state]
            traces.append(go.Scatter3d(
                x=cell_data['x'], y=cell_data['y'], z=cell_data['z'],
                mode='markers',
                marker=dict(size=cell_data['size'], color=cell_data['color'],
                           opacity=cell_data['alpha']),
                name=cell_data['name'], hoverinfo='name'
            ))
        
        frames = []
        for state_data in self.simulation_states:
            frame_data = []
            
            if self.left_lung_surface_mm is not None:
                frame_data.append(go.Scatter3d(
                    x=self.left_lung_surface_mm[:, 2],
                    y=self.left_lung_surface_mm[:, 1],
                    z=self.left_lung_surface_mm[:, 0],
                ))
            else:
                frame_data.append(go.Scatter3d(x=[], y=[], z=[]))
                
            if self.right_lung_surface_mm is not None:
                frame_data.append(go.Scatter3d(
                    x=self.right_lung_surface_mm[:, 2],
                    y=self.right_lung_surface_mm[:, 1],
                    z=self.right_lung_surface_mm[:, 0],
                ))
            else:
                frame_data.append(go.Scatter3d(x=[], y=[], z=[]))
            
            for state in cell_order:
                cell_data = state_data['cells'][state]
                frame_data.append(go.Scatter3d(
                    x=cell_data['x'], y=cell_data['y'], z=cell_data['z'],
                ))
            
            frames.append(go.Frame(
                data=frame_data,
                name=str(state_data['iteration']),
                traces=[0, 1, 2, 3, 4, 5]
            ))
        
        fig = go.Figure(data=traces, frames=frames)
        
        updatemenus = [
            dict(
                type="buttons",
                showactive=False,
                y=1.02,
                x=0.0,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[None, {
                            "frame": {"duration": 300, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 100}
                        }]
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[[None], {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }]
                    ),
                    dict(
                        label="⏮ Reset",
                        method="animate",
                        args=[[str(self.simulation_states[0]['iteration'])], {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }]
                    )
                ]
            ),
        ]
        
        sliders = [
            dict(
                active=0,
                yanchor="top",
                xanchor="left",
                currentvalue=dict(
                    font=dict(size=14),
                    prefix="Iteration: ",
                    visible=True,
                    xanchor="center"
                ),
                transition=dict(duration=100),
                pad=dict(b=10, t=50),
                len=0.9,
                x=0.05,
                y=0,
                steps=[
                    dict(
                        args=[[str(state['iteration'])],
                              dict(frame=dict(duration=0, redraw=True),
                                   mode="immediate",
                                   transition=dict(duration=0))],
                        label=str(state['iteration']),
                        method="animate"
                    )
                    for state in self.simulation_states
                ]
            )
        ]
        
        fig.update_layout(
            title=dict(
                text=f"<b>🫁 {patient_id} - {risk_label}</b><br>" +
                     f"<sup>α={p_alpha:.4f} | β={p_necrosis:.4f}</sup>",
                x=0.5,
                font=dict(size=18)
            ),
            scene=dict(
                xaxis=dict(title='X (mm)', backgroundcolor='rgb(240,245,250)',
                          gridcolor='white', showbackground=True),
                yaxis=dict(title='Y (mm)', backgroundcolor='rgb(240,245,250)',
                          gridcolor='white', showbackground=True),
                zaxis=dict(title='Z (mm)', backgroundcolor='rgb(240,245,250)',
                          gridcolor='white', showbackground=True),
                aspectmode='data',
                camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
            ),
            width=1100,
            height=850,
            updatemenus=updatemenus,
            sliders=sliders,
            legend=dict(
                yanchor="top", y=0.95,
                xanchor="right", x=0.99,
                bgcolor="rgba(255,255,255,0.9)",
                bordercolor="gray", borderwidth=1
            ),
            margin=dict(l=0, r=0, t=100, b=120)
        )
        
        html_content = fig.to_html(include_plotlyjs=True, full_html=True)
        
        custom_controls = """
<style>
    .control-panel {
        position: fixed;
        top: 10px;
        right: 10px;
        background: rgba(255,255,255,0.95);
        padding: 15px 20px;
        border-radius: 12px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        z-index: 1000;
        font-family: 'Segoe UI', Arial, sans-serif;
        min-width: 200px;
    }
    .control-panel h3 {
        margin: 0 0 12px 0;
        font-size: 14px;
        color: #333;
        border-bottom: 1px solid #eee;
        padding-bottom: 8px;
    }
    .slider-row {
        display: flex;
        align-items: center;
        margin: 10px 0;
    }
    .slider-row label {
        flex: 1;
        font-size: 13px;
        color: #555;
    }
    .slider-row input[type="range"] {
        width: 100px;
        cursor: pointer;
    }
    .slider-value {
        width: 45px;
        text-align: right;
        font-family: 'Consolas', monospace;
        font-size: 12px;
        color: #666;
    }
    .legend-section {
        margin-top: 15px;
        padding-top: 12px;
        border-top: 1px solid #eee;
    }
    .legend-section h4 {
        margin: 0 0 8px 0;
        font-size: 12px;
        color: #888;
    }
    .legend-item {
        display: flex;
        align-items: center;
        margin: 4px 0;
        font-size: 11px;
    }
    .legend-dot {
        width: 10px;
        height: 10px;
        border-radius: 50%;
        margin-right: 8px;
        flex-shrink: 0;
    }
</style>

<div class="control-panel">
    <h3>🎛️ Display Controls</h3>
    
    <div class="slider-row">
        <label>🫁 Lung Opacity</label>
        <input type="range" id="lungOpacity" min="0" max="50" value="15" 
               oninput="updateLungOpacity(this.value)">
        <span class="slider-value" id="opacityValue">0.15</span>
    </div>
    
    <div class="legend-section">
        <h4>LEGEND</h4>
        <div class="legend-item">
            <div class="legend-dot" style="background: dodgerblue;"></div>
            <span>Left Lung</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: limegreen;"></div>
            <span>Right Lung</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: darkblue;"></div>
            <span>Tumor Core</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: red;"></div>
            <span>Proliferating</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: purple;"></div>
            <span>Malignant</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: dimgray;"></div>
            <span>Necrotic</span>
        </div>
    </div>
</div>

<script>
function updateLungOpacity(value) {
    var opacity = value / 100;
    document.getElementById('opacityValue').textContent = opacity.toFixed(2);
    
    var plotDiv = document.getElementsByClassName('plotly-graph-div')[0];
    if (plotDiv) {
        Plotly.restyle(plotDiv, {'marker.opacity': opacity}, [0, 1]);
    }
}

document.addEventListener('DOMContentLoaded', function() {
    setTimeout(function() {
        updateLungOpacity(15);
    }, 500);
});
</script>
"""
        
        html_content = html_content.replace('</body>', custom_controls + '</body>')
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print(f"   ✅ Đã lưu: {output_path}")

    def run(self, lung_mask, tumor_mask, ct_volume, spacing, origin,
            p_alpha, p_necrosis, patient_id, risk_label) -> np.ndarray:
        """Chạy mô phỏng và tạo visualization."""
        print(f"\n🔄 Mô phỏng 3D cho {patient_id} ({risk_label})...")
        print(f"   📊 Tham số: α={p_alpha:.4f}, β={p_necrosis:.4f}")
        
        self.simulation_states = []
        
        self.transform = RealSpaceTransform(spacing, origin, lung_mask.shape, self.grid_size)

        lung_grid = self.resample(lung_mask, lung_mask.shape)
        tumor_grid = self.resample(tumor_mask, tumor_mask.shape)
        tumor_grid = tumor_grid & lung_grid

        self.prepare_lung_surfaces(lung_grid)

        grid = np.zeros_like(lung_grid, dtype=np.int8)
        grid[lung_grid] = 1
        grid[tumor_grid] = 2

        scale = np.array([self.grid_size] * 3) / np.array(ct_volume.shape)
        ct_small = zoom(ct_volume, scale, order=1)
        microenv = np.clip((ct_small + 1000) / 1400, 0, 1)

        self.capture_state(grid, 0)

        for it in range(self.cfg['num_iterations']):
            tumor_cells = (grid == 2) | (grid == 4) | (grid == 6)
            
            dilated = binary_dilation(tumor_cells)
            boundary = dilated & (grid == 1) & lung_grid
            candidates = np.argwhere(boundary)

            if len(candidates) > 0:
                n_grow = int(len(candidates) * p_alpha)
                if n_grow > 0:
                    weights = microenv[tuple(candidates.T)]
                    weights /= (weights.sum() + 1e-9)
                    indices = np.random.choice(len(candidates), 
                                              size=min(n_grow, len(candidates)),
                                              p=weights, replace=False)
                    for idx in indices:
                        grid[tuple(candidates[idx])] = 4

            mask_4 = (grid == 4)
            grid[mask_4 & (np.random.rand(*grid.shape) < 0.2)] = 2

            mask_2 = (grid == 2)
            grid[mask_2 & (np.random.rand(*grid.shape) < self.cfg['mutation_chance'])] = 6

            if it > self.cfg.get('necrosis_delay', 15):
                eroded = binary_erosion(tumor_cells, iterations=2)
                grid[eroded & (grid == 2) & (np.random.rand(*grid.shape) < p_necrosis)] = 3

            grid[~lung_grid] = 0

            if (it + 1) % self.cfg['capture_interval'] == 0:
                print(f"   📸 Iteration {it+1}/{self.cfg['num_iterations']}")
                self.capture_state(grid, it + 1)

        html_path = OUTPUT_DIR / f"{patient_id}_{risk_label}_digital_twin.html"
        self.create_interactive_html(patient_id, risk_label, p_alpha, p_necrosis, html_path)
        
        return grid


# ============================================================================
# 4. MAIN PIPELINE
# ============================================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "=" * 80)
    print("🚀 RADIOMICS-DRIVEN DIGITAL TWIN + INTERACTIVE VISUALIZATION")
    print(f"   Mode: {'DEBUG (5 patients)' if DEBUG_MODE else 'FULL RUN'}")
    print("=" * 80)
    
    # ========================================================================
    # PHASE 1: Load Clinical Data
    # ========================================================================
    if not CLINICAL_FILE.exists():
        print(f"❌ Error: Không tìm thấy file: {CLINICAL_FILE}")
        return

    df_clin = pd.read_csv(CLINICAL_FILE)
    df_clin.columns = [c.strip() for c in df_clin.columns]

    col_map = {
        'PatientID': 'PatientID', 'age': 'age',
        'Survival.time': 'time', 'deadstatus.event': 'event', 'gender': 'gender'
    }
    df_clin = df_clin.rename(columns=col_map)
    df_clin['PatientID'] = df_clin['PatientID'].astype(str)

    if 'gender' in df_clin.columns:
        df_clin['gender'] = df_clin['gender'].astype(str).str.lower().map(
            {'male': 1, 'female': 0}).fillna(1)
    else:
        df_clin['gender'] = 1

    df_clin['time'] = pd.to_numeric(df_clin['time'], errors='coerce')
    df_clin['event'] = pd.to_numeric(df_clin['event'], errors='coerce')
    df_clin = df_clin.dropna(subset=['time', 'event', 'age'])

    patient_list = df_clin['PatientID'].unique().tolist()
    if DEBUG_MODE:
        patient_list = patient_list[:5]

    # ========================================================================
    # PHASE 2: Batch Simulation
    # ========================================================================
    print(f"\n📊 PHASE 2: Batch Simulation ({len(patient_list)} patients)...")
    
    results = Parallel(n_jobs=N_JOBS)(
        delayed(process_wrapper)(pid) for pid in tqdm(patient_list, desc="Simulating")
    )

    valid_results = [r for r in results if r is not None]
    print(f"\n✅ Complete: {len(valid_results)} valid patients")

    if len(valid_results) <= 10:
        print("❌ Không đủ dữ liệu để phân tích.")
        return

    # ========================================================================
    # Print Biological Parameters for All Patients
    # ========================================================================
    print("\n" + "=" * 80)
    print("📋 THAM SỐ SINH HỌC CHO TẤT CẢ BỆNH NHÂN")
    print("=" * 80)
    
    params_df = pd.DataFrame(valid_results)
    params_df = params_df.sort_values('Bio_Alpha', ascending=False)
    
    alpha_unique = params_df['Bio_Alpha'].nunique()
    alpha_std = params_df['Bio_Alpha'].std()
    
    print(f"\n✅ Validation Check:")
    print(f"   Unique α values: {alpha_unique}/{len(params_df)}")
    print(f"   α std deviation: {alpha_std:.6f}")
    
    if alpha_unique == 1:
        print("   ⚠️ WARNING: All α values are identical! Formula may need adjustment.")
    elif alpha_std < 0.01:
        print("   ⚠️ WARNING: Very low α variation! Formula may need adjustment.")
    else:
        print("   ✅ Good diversity in α values!")
    
    print(f"\n{'No.':<5} {'PatientID':<20} {'α (Proliferation)':<20} {'β (Necrosis)':<20} {'Entropy':<12} {'Sphericity':<12}")
    print("=" * 89)
    
    for idx, row in enumerate(params_df.itertuples(), 1):
        print(f"{idx:<5} {row.PatientID:<20} {row.Bio_Alpha:<20.6f} {row.Bio_Necrosis:<20.6f} {row.Radiomics_Entropy:<12.4f} {row.Radiomics_Sphericity:<12.4f}")
    
    print("\n" + "=" * 80)
    print("📊 THỐNG KÊ TỔNG QUAN:")
    print("=" * 80)
    print(f"   α (Proliferation Rate):")
    print(f"      Mean:   {params_df['Bio_Alpha'].mean():.6f}")
    print(f"      Median: {params_df['Bio_Alpha'].median():.6f}")
    print(f"      Std:    {params_df['Bio_Alpha'].std():.6f}")
    print(f"      Range:  [{params_df['Bio_Alpha'].min():.6f}, {params_df['Bio_Alpha'].max():.6f}]")
    print(f"\n   β (Necrosis Rate):")
    print(f"      Mean:   {params_df['Bio_Necrosis'].mean():.6f}")
    print(f"      Median: {params_df['Bio_Necrosis'].median():.6f}")
    print(f"      Std:    {params_df['Bio_Necrosis'].std():.6f}")
    print(f"      Range:  [{params_df['Bio_Necrosis'].min():.6f}, {params_df['Bio_Necrosis'].max():.6f}]")
    
    params_output = OUTPUT_DIR / "all_patients_biological_parameters.csv"
    params_df_save = params_df[['PatientID', 'Bio_Alpha', 'Bio_Necrosis', 
                                 'Radiomics_Entropy', 'Radiomics_Sphericity',
                                 'Sim_Growth_Rate', 'Sim_Necrosis_Ratio']]
    params_df_save.to_csv(params_output, index=False)
    print(f"\n✅ Đã lưu tham số vào: {params_output}")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax1 = axes[0, 0]
    ax1.hist(params_df['Bio_Alpha'], bins=30, color='red', alpha=0.7, edgecolor='black')
    ax1.axvline(params_df['Bio_Alpha'].mean(), color='darkred', linestyle='--', 
                linewidth=2, label=f'Mean: {params_df["Bio_Alpha"].mean():.4f}')
    ax1.axvline(params_df['Bio_Alpha'].median(), color='orange', linestyle='-.', 
                linewidth=2, label=f'Median: {params_df["Bio_Alpha"].median():.4f}')
    ax1.set_xlabel('α (Proliferation Rate)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax1.set_title('Distribution of Proliferation Rate (α)', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[0, 1]
    ax2.hist(params_df['Bio_Necrosis'], bins=30, color='purple', alpha=0.7, edgecolor='black')
    ax2.axvline(params_df['Bio_Necrosis'].mean(), color='darkviolet', linestyle='--', 
                linewidth=2, label=f'Mean: {params_df["Bio_Necrosis"].mean():.4f}')
    ax2.axvline(params_df['Bio_Necrosis'].median(), color='magenta', linestyle='-.', 
                linewidth=2, label=f'Median: {params_df["Bio_Necrosis"].median():.4f}')
    ax2.set_xlabel('β (Necrosis Rate)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of Necrosis Rate (β)', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    ax3 = axes[1, 0]
    scatter = ax3.scatter(params_df['Bio_Alpha'], params_df['Bio_Necrosis'], 
                         c=params_df['Radiomics_Entropy'], cmap='viridis', 
                         s=100, alpha=0.6, edgecolor='black')
    ax3.set_xlabel('α (Proliferation Rate)', fontsize=12, fontweight='bold')
    ax3.set_ylabel('β (Necrosis Rate)', fontsize=12, fontweight='bold')
    ax3.set_title('α vs β Relationship (colored by Entropy)', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax3)
    cbar.set_label('Entropy', fontsize=10, fontweight='bold')
    
    ax4 = axes[1, 1]
    top10 = params_df.head(10)
    y_pos = np.arange(len(top10))
    ax4.barh(y_pos, top10['Bio_Alpha'].values, color='crimson', alpha=0.7, edgecolor='black')
    ax4.set_yticks(y_pos)
    ax4.set_yticklabels([pid[:15] + '...' if len(pid) > 15 else pid 
                         for pid in top10['PatientID'].values], fontsize=9)
    ax4.set_xlabel('α (Proliferation Rate)', fontsize=12, fontweight='bold')
    ax4.set_title('Top 10 Patients by Proliferation Rate', fontsize=13, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='x')
    ax4.invert_yaxis()
    
    plt.tight_layout()
    params_plot = OUTPUT_DIR / "biological_parameters_distribution.png"
    plt.savefig(params_plot, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Đã lưu biểu đồ phân bố: {params_plot}")
    print("=" * 80 + "\n")

    # ========================================================================
    # PHASE 3: Cox Survival Analysis + Stratified K-Fold CV
    # ========================================================================
    print("\n📊 PHASE 3: Cox Survival Analysis + Cross-Validation...")

    sim_df = pd.DataFrame(valid_results).set_index('PatientID')
    df_clin_indexed = df_clin.set_index('PatientID')
    final_df = df_clin_indexed.join(sim_df, how='inner')
    final_df['time_years'] = final_df['time'] / 365.25

    feature_list = ['Sim_Growth_Rate', 'Sim_Necrosis_Ratio', 'age', 'gender']
    final_df = final_df.dropna(subset=feature_list + ['time_years', 'event'])

    scaler = StandardScaler()
    df_scaled = final_df.copy()
    df_scaled[feature_list] = scaler.fit_transform(final_df[feature_list])

    cph = CoxPHFitter()
    cph.fit(df_scaled[feature_list + ['time_years', 'event']],
            duration_col='time_years', event_col='event')

    print(f"\n🔹 Full Model C-Index: {cph.concordance_index_:.4f}")
    print(f"\n📋 Cox Model Summary:")
    print(cph.summary[['coef', 'exp(coef)', 'p']])

    # Stratified K-Fold Cross-Validation
    print("\n" + "=" * 60)
    print("📊 STRATIFIED K-FOLD CROSS-VALIDATION")
    print("=" * 60)
    
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    stratify_labels = final_df['event'].values.astype(int)
    
    cv_scores = []
    fold_results = []
    
    print(f"\nRunning {n_splits}-Fold Cross-Validation...")
    print(f"Total samples: {len(df_scaled)} | Events: {int(final_df['event'].sum())}\n")
    
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df_scaled, stratify_labels), 1):
        train_data = df_scaled.iloc[train_idx]
        test_data = df_scaled.iloc[test_idx]
        
        cph_fold = CoxPHFitter()
        cph_fold.fit(train_data[feature_list + ['time_years', 'event']],
                     duration_col='time_years', event_col='event')
        
        c_index_test = concordance_index(
            test_data['time_years'],
            -cph_fold.predict_partial_hazard(test_data[feature_list]),
            test_data['event']
        )
        
        cv_scores.append(c_index_test)
        fold_results.append({
            'Fold': fold_idx,
            'Train_Size': len(train_idx),
            'Test_Size': len(test_idx),
            'Train_Events': int(train_data['event'].sum()),
            'Test_Events': int(test_data['event'].sum()),
            'C_Index': c_index_test
        })
        
        print(f"   Fold {fold_idx}/{n_splits}: C-Index = {c_index_test:.4f} " +
              f"(Train: {len(train_idx)}, Test: {len(test_idx)}, " +
              f"Test Events: {int(test_data['event'].sum())})")
    
    cv_mean = np.mean(cv_scores)
    cv_std = np.std(cv_scores)
    cv_min = np.min(cv_scores)
    cv_max = np.max(cv_scores)
    
    print(f"\n{'='*60}")
    print(f"📊 CROSS-VALIDATION SUMMARY:")
    print(f"{'='*60}")
    print(f"   Mean C-Index:     {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"   Range:            [{cv_min:.4f}, {cv_max:.4f}]")
    print(f"   Full Model:       {cph.concordance_index_:.4f}")
    print(f"   Performance:      {'✅ Stable' if cv_std < 0.05 else '⚠️ Variable'}")
    print(f"{'='*60}\n")
    
    cv_results_df = pd.DataFrame(fold_results)
    cv_path = OUTPUT_DIR / "cross_validation_results.csv"
    cv_results_df.to_csv(cv_path, index=False)
    print(f"✅ Saved CV results: {cv_path}")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.bar(range(1, n_splits + 1), cv_scores, color='steelblue', alpha=0.7, 
            edgecolor='black', linewidth=1.5)
    ax1.axhline(y=cv_mean, color='red', linestyle='--', linewidth=2, 
                label=f'CV Mean: {cv_mean:.4f}')
    ax1.axhline(y=cph.concordance_index_, color='green', linestyle='-.', linewidth=2, 
                label=f'Full Model: {cph.concordance_index_:.4f}')
    ax1.set_xlabel('Fold', fontsize=12, fontweight='bold')
    ax1.set_ylabel('C-Index', fontsize=12, fontweight='bold')
    ax1.set_title('Cross-Validation C-Index by Fold', fontsize=13, fontweight='bold')
    ax1.set_ylim([max(0, cv_min - 0.05), min(1, cv_max + 0.05)])
    ax1.set_xticks(range(1, n_splits + 1))
    ax1.legend(fontsize=10, loc='lower right')
    ax1.grid(True, alpha=0.3, linestyle='--')
    
    for i, v in enumerate(cv_scores, 1):
        ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', 
                fontsize=9, fontweight='bold')
    
    ax2.boxplot([cv_scores], labels=['CV C-Index'], 
                patch_artist=True, 
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2),
                widths=0.5)
    ax2.scatter([1]*len(cv_scores), cv_scores, color='darkblue', s=100, 
                alpha=0.6, zorder=3, label='Fold Results')
    ax2.axhline(y=cph.concordance_index_, color='green', linestyle='-.', 
                linewidth=2, label=f'Full Model: {cph.concordance_index_:.4f}')
    ax2.set_ylabel('C-Index', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of CV C-Index Scores', fontsize=13, fontweight='bold')
    ax2.set_ylim([max(0, cv_min - 0.05), min(1, cv_max + 0.05)])
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    stats_text = f'Mean: {cv_mean:.4f}\nStd: {cv_std:.4f}\nMin: {cv_min:.4f}\nMax: {cv_max:.4f}'
    ax2.text(0.98, 0.02, stats_text, transform=ax2.transAxes,
            fontsize=9, verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    cv_plot_path = OUTPUT_DIR / "cross_validation_plot.png"
    plt.savefig(cv_plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved CV plot: {cv_plot_path}\n")

    # ========================================================================
    # PHASE 4: Identify Representative Patients (2 High + 2 Low)
    # ========================================================================
    print("\n" + "=" * 60)
    print("🔬 PHASE 4: CASE STUDY - 2 HIGH RISK + 2 LOW RISK")
    print("=" * 60)

    risk_scores = cph.predict_partial_hazard(df_scaled)
    final_df['Risk_Score'] = risk_scores

    # Get top 2 highest risk and bottom 2 lowest risk
    risk_sorted = final_df.sort_values('Risk_Score', ascending=False)
    # Chia làm 2 nhóm: high risk (trên median) và low risk (dưới median)
    median_risk = risk_sorted['Risk_Score'].median()
    high_risk_pool = risk_sorted[risk_sorted['Risk_Score'] > median_risk]
    low_risk_pool = risk_sorted[risk_sorted['Risk_Score'] <= median_risk]

    # Random sample 5 từ mỗi nhóm
    top_2_high = high_risk_pool.sample(n=min(5, len(high_risk_pool)), random_state=42)
    top_2_low = low_risk_pool.sample(n=min(5, len(low_risk_pool)), random_state=42)

    def print_profile(row, label, pid):
        print(f"\n📌 {label} (Patient: {pid})")
        print(f"   Survival: {row['time_years']:.2f} yrs | " +
              f"Event: {'Dead' if row['event'] == 1 else 'Censored'}")
        print(f"   Age: {row['age']:.0f} | Gender: {'M' if row['gender'] == 1 else 'F'}")
        print(f"   Risk Score: {row['Risk_Score']:.4f}")
        print(f"   α (prolif): {row['Bio_Alpha']:.4f} | β (necro): {row['Bio_Necrosis']:.4f}")

    print("\n🔴 HIGH RISK PATIENTS:")
    for i, (pid, row) in enumerate(top_2_high.iterrows(), 1):
        print_profile(row, f"HIGH RISK #{i}", pid)

    print("\n🟢 LOW RISK PATIENTS:")
    for i, (pid, row) in enumerate(top_2_low.iterrows(), 1):
        print_profile(row, f"LOW RISK #{i}", pid)

    # ========================================================================
    # PHASE 5: Interactive 3D Visualization (4 patients)
    # ========================================================================
    print("\n" + "=" * 60)
    print("🎨 PHASE 5: INTERACTIVE 3D VISUALIZATION (4 patients)")
    print("=" * 60)

    visualization_patients = []
    for i, (pid, row) in enumerate(top_2_high.iterrows(), 1):
        visualization_patients.append((pid, row, f"HIGH_RISK_{i}"))
    for i, (pid, row) in enumerate(top_2_low.iterrows(), 1):
        visualization_patients.append((pid, row, f"LOW_RISK_{i}"))

    for pid, row, label in visualization_patients:
        print(f"\n{'='*40}")
        
        data = load_patient_data_smart(pid, LUNG1_ROOT)
        if data is None:
            print(f"   ⚠️ Không load được data cho {pid}")
            continue

        vol, tumor, lung, spacing, origin = data

        visualizer = MLPA3DVisualizer(MLPA_CONFIG_VIZ)
        visualizer.run(
            lung_mask=lung,
            tumor_mask=tumor,
            ct_volume=vol,
            spacing=spacing,
            origin=origin,
            p_alpha=row['Bio_Alpha'],
            p_necrosis=row['Bio_Necrosis'],
            patient_id=pid,
            risk_label=label
        )

    # ========================================================================
    # PHASE 6: Kaplan-Meier Plot
    # ========================================================================
    print("\n📊 PHASE 6: Kaplan-Meier Survival Curves...")

    plt.figure(figsize=(10, 6))
    
    med_score = risk_scores.median()
    high_group = final_df[risk_scores > med_score]
    low_group = final_df[risk_scores <= med_score]

    kmf = KaplanMeierFitter()
    
    kmf.fit(high_group['time_years'], high_group['event'], label='High Risk')
    kmf.plot_survival_function(color='red', linewidth=2)
    
    kmf.fit(low_group['time_years'], low_group['event'], label='Low Risk')
    kmf.plot_survival_function(color='blue', linewidth=2)

    p_val = logrank_test(
        high_group['time_years'], low_group['time_years'],
        event_observed_A=high_group['event'], event_observed_B=low_group['event']
    ).p_value

    plt.title(f"Digital Twin Prognosis (Log-Rank p={p_val:.5f})", fontsize=14)
    plt.xlabel("Years", fontsize=12)
    plt.ylabel("Survival Probability", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)

    km_path = OUTPUT_DIR / "kaplan_meier_survival.png"
    plt.savefig(km_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: {km_path}")

    # ========================================================================
    # Summary
    # ========================================================================
    print("\n" + "=" * 60)
    print("✅ HOÀN THÀNH!")
    print("=" * 60)
    print(f"\n📁 Output Directory: {OUTPUT_DIR}/")
    print(f"\n📊 Analysis Results:")
    print(f"   • all_patients_biological_parameters.csv")
    print(f"   • biological_parameters_distribution.png")
    print(f"   • cross_validation_results.csv")
    print(f"   • cross_validation_plot.png")
    print(f"   • kaplan_meier_survival.png")
    print(f"\n🎨 3D Visualizations (4 files):")
    for pid, _, label in visualization_patients:
        print(f"   • {pid}_{label}_digital_twin.html")
    print(f"\n💡 Interactive Features:")
    print(f"   - ▶ Play / ⏸ Pause: Control animation")
    print(f"   - 🎚️ Lung Opacity: Adjust transparency")
    print(f"   - 🖱️ Mouse: Rotate and zoom 3D model")
    print(f"\n📈 Model Performance:")
    print(f"   - Full Model C-Index: {cph.concordance_index_:.4f}")
    print(f"   - CV Mean C-Index: {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"   - Model Stability: {'✅ Good' if cv_std < 0.05 else '⚠️ Check results'}")
    print(f"\n🧬 Biological Parameters:")
    print(f"   - Total Patients: {len(valid_results)}")
    print(f"   - α Range: [{params_df['Bio_Alpha'].min():.4f}, {params_df['Bio_Alpha'].max():.4f}]")
    print(f"   - β Range: [{params_df['Bio_Necrosis'].min():.4f}, {params_df['Bio_Necrosis'].max():.4f}]")
    print(f"\n🔬 Case Study: 4 Representative Patients")
    print(f"   - 2 High Risk patients visualized")
    print(f"   - 2 Low Risk patients visualized")


if __name__ == "__main__":
    main()


🚀 RADIOMICS-DRIVEN DIGITAL TWIN + INTERACTIVE VISUALIZATION
   Mode: FULL RUN

📊 PHASE 2: Batch Simulation (400 patients)...


Simulating: 100%|██████████| 400/400 [04:26<00:00,  1.50it/s]



✅ Complete: 390 valid patients

📋 THAM SỐ SINH HỌC CHO TẤT CẢ BỆNH NHÂN

✅ Validation Check:
   Unique α values: 382/390
   α std deviation: 0.010562
   ✅ Good diversity in α values!

No.   PatientID            α (Proliferation)    β (Necrosis)         Entropy      Sphericity  
1     LUNG1-386            0.060000             0.045167             0.8173       0.5604      
2     LUNG1-296            0.054764             0.052302             0.7372       0.4712      
3     LUNG1-282            0.054372             0.040363             0.7325       0.6205      
4     LUNG1-250            0.052989             0.034350             0.7159       0.6956      
5     LUNG1-167            0.052403             0.054249             0.7088       0.4469      
6     LUNG1-086            0.052251             0.043344             0.7070       0.5832      
7     LUNG1-415            0.051541             0.037787             0.6985       0.6527      
8     LUNG1-094            0.051299             0.03442

✅ Đã lưu biểu đồ phân bố: digital_twin_output/biological_parameters_distribution.png


📊 PHASE 3: Cox Survival Analysis + Cross-Validation...

🔹 Full Model C-Index: 0.5856

📋 Cox Model Summary:
                        coef  exp(coef)         p
covariate                                        
Sim_Growth_Rate     0.037940   1.038668  0.488770
Sim_Necrosis_Ratio  0.223443   1.250374  0.000059
age                 0.197352   1.218173  0.001364
gender              0.076036   1.079002  0.182508

📊 STRATIFIED K-FOLD CROSS-VALIDATION

Running 5-Fold Cross-Validation...
Total samples: 390 | Events: 345

   Fold 1/5: C-Index = 0.5812 (Train: 312, Test: 78, Test Events: 69)
   Fold 2/5: C-Index = 0.5428 (Train: 312, Test: 78, Test Events: 69)
   Fold 3/5: C-Index = 0.5282 (Train: 312, Test: 78, Test Events: 69)
   Fold 4/5: C-Index = 0.6234 (Train: 312, Test: 78, Test Events: 69)
   Fold 5/5: C-Index = 0.6175 (Train: 312, Test: 78, Test Events: 69)

📊 CROSS-VALIDATION SUMMARY:
   Mean C-Index:   


🔄 Mô phỏng 3D cho LUNG1-243 (HIGH_RISK_4)...
   📊 Tham số: α=0.0411, β=0.0418
   🔨 Đang tách và trích xuất bề mặt 2 lá phổi...
      - Left Lung: 1 voxels → 1 points
      - Right Lung: 9119 voxels → 1520 points
   📸 Iteration 3/150
   📸 Iteration 6/150
   📸 Iteration 9/150
   📸 Iteration 12/150
   📸 Iteration 15/150
   📸 Iteration 18/150
   📸 Iteration 21/150
   📸 Iteration 24/150
   📸 Iteration 27/150
   📸 Iteration 30/150
   📸 Iteration 33/150
   📸 Iteration 36/150
   📸 Iteration 39/150
   📸 Iteration 42/150
   📸 Iteration 45/150
   📸 Iteration 48/150
   📸 Iteration 51/150
   📸 Iteration 54/150
   📸 Iteration 57/150
   📸 Iteration 60/150
   📸 Iteration 63/150
   📸 Iteration 66/150
   📸 Iteration 69/150
   📸 Iteration 72/150
   📸 Iteration 75/150
   📸 Iteration 78/150
   📸 Iteration 81/150
   📸 Iteration 84/150
   📸 Iteration 87/150
   📸 Iteration 90/150
   📸 Iteration 93/150
   📸 Iteration 96/150
   📸 Iteration 99/150
   📸 Iteration 102/150
   📸 Iteration 105/150
   📸 Iteration 108

   📸 Iteration 51/150
   📸 Iteration 54/150
   📸 Iteration 57/150
   📸 Iteration 60/150
   📸 Iteration 63/150
   📸 Iteration 66/150
   📸 Iteration 69/150
   📸 Iteration 72/150
   📸 Iteration 75/150
   📸 Iteration 78/150
   📸 Iteration 81/150
   📸 Iteration 84/150
   📸 Iteration 87/150
   📸 Iteration 90/150
   📸 Iteration 93/150
   📸 Iteration 96/150
   📸 Iteration 99/150
   📸 Iteration 102/150
   📸 Iteration 105/150
   📸 Iteration 108/150
   📸 Iteration 111/150
   📸 Iteration 114/150
   📸 Iteration 117/150
   📸 Iteration 120/150
   📸 Iteration 123/150
   📸 Iteration 126/150
   📸 Iteration 129/150
   📸 Iteration 132/150
   📸 Iteration 135/150
   📸 Iteration 138/150
   📸 Iteration 141/150
   📸 Iteration 144/150
   📸 Iteration 147/150
   📸 Iteration 150/150
   🌐 Đang tạo HTML interactive...
   ✅ Đã lưu: digital_twin_output/LUNG1-166_LOW_RISK_4_digital_twin.html


🔄 Mô phỏng 3D cho LUNG1-102 (LOW_RISK_5)...
   📊 Tham số: α=0.0414, β=0.0443
   🔨 Đang tách và trích xuất bề mặt 2 lá phổi...
 

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
RADIOMICS-DRIVEN DIGITAL TWIN + INTERACTIVE 3D VISUALIZATION
================================================================================
Pipeline kết hợp:
1. Batch Simulation & Cox Survival Analysis with Stratified K-Fold CV
2. Print α, β parameters for ALL patients
3. Case Study: Tìm 4 bệnh nhân đặc trưng (2 High Risk + 2 Low Risk)
4. Interactive HTML: Animation với Play/Stop + Opacity Slider

Author: Paul Nguyen
Modified: 4 representative patients (2 high + 2 low risk)
"""

import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from scipy.ndimage import zoom, binary_erosion, binary_dilation
from scipy.ndimage import label as ndimage_label
from scipy import ndimage
from skimage import measure
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# 1. CẤU HÌNH HỆ THỐNG
# ============================================================================

LUNG1_ROOT = Path("./nsclc/manifest-1603198545583/NSCLC-Radiomics")
CLINICAL_FILE = Path("./nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv")
OUTPUT_DIR = Path("./digital_twin_output")

N_JOBS = -1
DEBUG_MODE = False

# Tham số cho Batch Simulation (nhanh)
MLPA_CONFIG_BATCH = {
    'grid_size': 90,
    'num_iterations': 120,
    'mutation_chance': 0.005,
}

# Tham số cho 3D Visualization (chi tiết hơn)
MLPA_CONFIG_VIZ = {
    'grid_size': 90,
    'num_iterations': 120,
    'mutation_chance': 0.002,
    'necrosis_delay': 15,
    'capture_interval': 3,
}


# ============================================================================
# 2. DATA LOADING & RADIOMICS ENGINE
# ============================================================================

def load_patient_data_smart(patient_id: str, data_root: Path) -> Optional[Tuple]:
    """Load CT và Segmentation từ DICOM."""
    p_dir = data_root / patient_id
    if not p_dir.exists():
        return None

    ct_dirs = []
    seg_file = None
    
    for study in p_dir.glob("*"):
        if not study.is_dir():
            continue
        for subdir in study.iterdir():
            if subdir.is_dir():
                if subdir.name.startswith("300."):
                    segs = list(subdir.glob("*.dcm"))
                    seg_file = segs[0] if segs else None
                else:
                    dcm_count = len(list(subdir.glob("*.dcm")))
                    if dcm_count > 10:
                        ct_dirs.append((subdir, dcm_count))

    if not ct_dirs or not seg_file:
        return None
    
    ct_dir = max(ct_dirs, key=lambda x: x[1])[0]

    try:
        slices = [pydicom.dcmread(str(f)) for f in sorted(ct_dir.glob("*.dcm"))]
        slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
        volume = np.stack([s.pixel_array for s in slices])
        slope = float(getattr(slices[0], 'RescaleSlope', 1))
        intercept = float(getattr(slices[0], 'RescaleIntercept', 0))
        volume = volume * slope + intercept

        pixel_spacing = [float(x) for x in slices[0].PixelSpacing]
        z_positions = [float(s.ImagePositionPatient[2]) for s in slices]
        z_spacing = abs(z_positions[1] - z_positions[0]) if len(z_positions) > 1 else 1.0
        spacing = np.array([z_spacing, pixel_spacing[0], pixel_spacing[1]])
        origin = np.array([
            float(slices[0].ImagePositionPatient[2]),
            float(slices[0].ImagePositionPatient[1]),
            float(slices[0].ImagePositionPatient[0])
        ])

        seg_ds = pydicom.dcmread(str(seg_file))
        sop_to_z = {ds.SOPInstanceUID: idx for idx, ds in enumerate(slices)}

        tumor_mask = np.zeros_like(volume, dtype=np.uint8)
        lung_mask = np.zeros_like(volume, dtype=np.uint8)
        
        frames = seg_ds.pixel_array
        if frames.ndim == 2:
            frames = frames[np.newaxis, ...]

        seg_info = {}
        if hasattr(seg_ds, 'SegmentSequence'):
            for item in seg_ds.SegmentSequence:
                seg_info[int(item.SegmentNumber)] = getattr(item, 'SegmentLabel', '').lower()

        if hasattr(seg_ds, 'PerFrameFunctionalGroupsSequence'):
            for i, meta in enumerate(seg_ds.PerFrameFunctionalGroupsSequence):
                try:
                    seg_num = int(meta.SegmentIdentificationSequence[0].ReferencedSegmentNumber)
                    ref_uid = meta.DerivationImageSequence[0].SourceImageSequence[0].ReferencedSOPInstanceUID
                    if ref_uid in sop_to_z:
                        z = sop_to_z[ref_uid]
                        mask_slice = (frames[i] > 0).astype(np.uint8)
                        name = seg_info.get(seg_num, '')
                        
                        if any(k in name for k in ['gtv', 'tumor', 'neoplasm']):
                            tumor_mask[z] = np.maximum(tumor_mask[z], mask_slice)
                        elif any(k in name for k in ['lung', 'left', 'right']):
                            lung_mask[z] = np.maximum(lung_mask[z], mask_slice)
                except:
                    continue

        if np.sum(tumor_mask) == 0 or np.sum(lung_mask) == 0:
            return None

        for z in range(lung_mask.shape[0]):
            lung_mask[z] = ndimage.binary_fill_holes(lung_mask[z])

        lung_mask = np.logical_or(lung_mask, tumor_mask).astype(np.uint8)

        return volume, tumor_mask, lung_mask, spacing, origin

    except Exception:
        return None


def calculate_radiomics_phenotype(ct_volume: np.ndarray, tumor_mask: np.ndarray) -> Optional[Tuple]:
    """Tính toán Radiomics và map sang tham số sinh học."""
    voxels = ct_volume[tumor_mask > 0]
    if len(voxels) == 0:
        return None
    
    try:
        hist, _ = np.histogram(voxels, bins=64, density=True)
        hist = hist[hist > 0]
        entropy = -np.sum(hist * np.log2(hist))

        verts, faces, _, _ = measure.marching_cubes(tumor_mask, level=0.5)
        area = measure.mesh_surface_area(verts, faces)
        vol = np.sum(tumor_mask)
        sphericity = (np.pi**(1/3) * (6 * vol)**(2/3)) / area if area > 0 else 0
    except:
        return None

    entropy_normalized = np.clip((entropy - 0.2) / 0.6, 0, 1)
    
    p_alpha = 0.01 + (0.05 * entropy_normalized)
    p_necrosis = 0.01 + (0.08 * (1.0 - np.clip(sphericity, 0, 1)))

    return p_alpha, p_necrosis, entropy, sphericity


def run_simulation_batch(ct_volume, tumor_mask, lung_mask, config) -> Optional[Dict]:
    """Chạy simulation nhanh cho batch processing."""
    params = calculate_radiomics_phenotype(ct_volume, tumor_mask)
    if params is None:
        return None
    
    p_alpha, p_necrosis, p_entropy, p_sphericity = params

    scale = np.array([config['grid_size']] * 3) / np.array(tumor_mask.shape)
    tumor_grid = zoom(tumor_mask.astype(float), scale, order=0) > 0.5
    lung_grid = zoom(lung_mask.astype(float), scale, order=0) > 0.5
    tumor_grid = tumor_grid & lung_grid
    
    if np.sum(tumor_grid) == 0:
        return None

    ct_small = zoom(ct_volume, scale, order=1)
    microenv = np.clip((ct_small + 1000) / 1400, 0, 1)

    grid = np.zeros_like(tumor_grid, dtype=np.int8)
    grid[lung_grid] = 1
    grid[tumor_grid] = 2
    init_count = np.sum(grid == 2)

    for it in range(config['num_iterations']):
        cells = (grid == 2) | (grid == 4) | (grid == 6)
        bound = binary_dilation(cells) & (grid == 1)
        cands = np.argwhere(bound)
        
        if len(cands) > 0:
            n_grow = int(len(cands) * p_alpha)
            if n_grow > 0:
                n_grow = min(n_grow, len(cands))
                w = microenv[tuple(cands.T)]
                w_sum = w.sum()
                prob = w / w_sum if w_sum > 0 else None
                idx = np.random.choice(len(cands), n_grow, p=prob, replace=False)
                for i in idx:
                    grid[tuple(cands[i])] = 4

        mask_4 = grid == 4
        grid[mask_4 & (np.random.rand(*grid.shape) < 0.2)] = 2
        
        mask_2 = grid == 2
        grid[mask_2 & (np.random.rand(*grid.shape) < config['mutation_chance'])] = 6
        
        if it > 15:
            ero = binary_erosion(cells, iterations=2)
            grid[ero & (grid == 2) & (np.random.rand(*grid.shape) < p_necrosis)] = 3
        
        grid[~lung_grid] = 0

    total = np.sum((grid == 2) | (grid == 4) | (grid == 6))
    if total == 0:
        return None

    return {
        'Sim_Growth_Rate': (total - init_count) / config['num_iterations'],
        'Sim_Necrosis_Ratio': np.sum(grid == 3) / total,
        'Radiomics_Entropy': p_entropy,
        'Radiomics_Sphericity': p_sphericity,
        'Bio_Alpha': p_alpha,
        'Bio_Necrosis': p_necrosis,
    }


def process_wrapper(pid: str) -> Optional[Dict]:
    """Wrapper cho parallel processing."""
    try:
        data = load_patient_data_smart(pid, LUNG1_ROOT)
        if data is None:
            return None
        vol, tumor, lung, spacing, origin = data
        feats = run_simulation_batch(vol, tumor, lung, MLPA_CONFIG_BATCH)
        if feats:
            feats['PatientID'] = pid
        return feats
    except:
        return None


# ============================================================================
# 3. INTERACTIVE 3D VISUALIZATION ENGINE  
# ============================================================================

class RealSpaceTransform:
    """Chuyển đổi tọa độ grid sang không gian thực (mm)."""
    
    def __init__(self, spacing, origin, original_shape, grid_size):
        self.spacing = spacing
        self.origin = origin
        self.scale = np.array(original_shape) / grid_size

    def grid_to_mm(self, grid_coords):
        return (grid_coords * self.scale) * self.spacing + self.origin


class MLPA3DVisualizer:
    """Mô phỏng 3D với Interactive HTML."""

    def __init__(self, config: Dict):
        self.cfg = config
        self.grid_size = config['grid_size']
        self.transform = None
        self.left_lung_surface_mm = None
        self.right_lung_surface_mm = None
        self.plot_limits = None
        self.simulation_states = []

    def resample(self, mask, shape):
        scale = np.array([self.grid_size] * 3) / np.array(shape)
        return zoom(mask.astype(float), scale, order=0) > 0.5

    def separate_lungs(self, lung_grid):
        """Tách 2 lá phổi dựa trên connected components."""
        labeled, num_features = ndimage_label(lung_grid)
        
        if num_features < 2:
            mid_x = lung_grid.shape[2] // 2
            left_lung = lung_grid.copy()
            right_lung = lung_grid.copy()
            left_lung[:, :, mid_x:] = False
            right_lung[:, :, :mid_x] = False
            return left_lung, right_lung
        
        regions = []
        for i in range(1, num_features + 1):
            region_mask = (labeled == i)
            region_size = np.sum(region_mask)
            coords = np.argwhere(region_mask)
            centroid_x = coords[:, 2].mean() if len(coords) > 0 else 0
            regions.append((i, region_size, centroid_x, region_mask))
        
        regions.sort(key=lambda x: x[1], reverse=True)
        top_regions = regions[:2]
        
        if top_regions[0][2] < top_regions[1][2]:
            right_lung = top_regions[0][3]
            left_lung = top_regions[1][3]
        else:
            left_lung = top_regions[0][3]
            right_lung = top_regions[1][3]
        
        return left_lung, right_lung

    def prepare_lung_surfaces(self, lung_grid):
        """Trích xuất bề mặt 2 lá phổi riêng biệt."""
        print("   🔨 Đang tách và trích xuất bề mặt 2 lá phổi...")
        
        left_lung, right_lung = self.separate_lungs(lung_grid)
        
        def extract_surface(lung_part, name):
            eroded = binary_erosion(lung_part, iterations=1)
            surface_grid = lung_part & ~eroded
            coords = np.argwhere(surface_grid)
            
            if len(coords) > 0:
                mm_coords = self.transform.grid_to_mm(coords)
                step = max(1, len(mm_coords) // 1500)
                print(f"      - {name}: {len(coords)} voxels → {len(mm_coords[::step])} points")
                return mm_coords[::step]
            return None
        
        self.left_lung_surface_mm = extract_surface(left_lung, "Left Lung")
        self.right_lung_surface_mm = extract_surface(right_lung, "Right Lung")
        
        all_coords = np.argwhere(lung_grid)
        if len(all_coords) > 0:
            mm_coords = self.transform.grid_to_mm(all_coords)
            self.plot_limits = {
                'x': (mm_coords[:, 2].min(), mm_coords[:, 2].max()),
                'y': (mm_coords[:, 1].min(), mm_coords[:, 1].max()),
                'z': (mm_coords[:, 0].min(), mm_coords[:, 0].max())
            }

    def capture_state(self, grid, iteration):
        """Lưu trạng thái simulation để animate."""
        state_data = {'iteration': iteration, 'cells': {}}
        
        cell_configs = [
            (2, 'darkblue', 'Tumor Core', 4, 0.7),
            (4, 'red', 'Proliferating', 6, 1.0),
            (6, 'purple', 'Malignant', 5, 0.9),
            (3, 'dimgray', 'Necrotic', 4, 0.5),
        ]
        
        for state, color, name, size, alpha in cell_configs:
            coords = np.argwhere(grid == state)
            if len(coords) > 0:
                mm = self.transform.grid_to_mm(coords)
                step = max(1, len(mm) // 1000)
                mm = mm[::step]
                state_data['cells'][state] = {
                    'x': mm[:, 2].tolist(),
                    'y': mm[:, 1].tolist(),
                    'z': mm[:, 0].tolist(),
                    'color': color, 'name': name,
                    'size': size, 'alpha': alpha
                }
            else:
                state_data['cells'][state] = {
                    'x': [], 'y': [], 'z': [],
                    'color': color, 'name': name,
                    'size': size, 'alpha': alpha
                }
        
        self.simulation_states.append(state_data)

    def create_interactive_html(self, patient_id, risk_label, p_alpha, p_necrosis, output_path):
        """Tạo HTML interactive với Animation + Opacity Slider."""
        print(f"   🌐 Đang tạo HTML interactive...")
        
        traces = []
        
        if self.left_lung_surface_mm is not None:
            traces.append(go.Scatter3d(
                x=self.left_lung_surface_mm[:, 2],
                y=self.left_lung_surface_mm[:, 1],
                z=self.left_lung_surface_mm[:, 0],
                mode='markers',
                marker=dict(size=2, color='dodgerblue', opacity=0.15),
                name='Left Lung', hoverinfo='name'
            ))
        else:
            traces.append(go.Scatter3d(x=[], y=[], z=[], mode='markers', name='Left Lung'))
        
        if self.right_lung_surface_mm is not None:
            traces.append(go.Scatter3d(
                x=self.right_lung_surface_mm[:, 2],
                y=self.right_lung_surface_mm[:, 1],
                z=self.right_lung_surface_mm[:, 0],
                mode='markers',
                marker=dict(size=2, color='limegreen', opacity=0.15),
                name='Right Lung', hoverinfo='name'
            ))
        else:
            traces.append(go.Scatter3d(x=[], y=[], z=[], mode='markers', name='Right Lung'))
        
        cell_order = [2, 4, 6, 3]
        initial_state = self.simulation_states[0]
        
        for state in cell_order:
            cell_data = initial_state['cells'][state]
            traces.append(go.Scatter3d(
                x=cell_data['x'], y=cell_data['y'], z=cell_data['z'],
                mode='markers',
                marker=dict(size=cell_data['size'], color=cell_data['color'],
                           opacity=cell_data['alpha']),
                name=cell_data['name'], hoverinfo='name'
            ))
        
        frames = []
        for state_data in self.simulation_states:
            frame_data = []
            
            if self.left_lung_surface_mm is not None:
                frame_data.append(go.Scatter3d(
                    x=self.left_lung_surface_mm[:, 2],
                    y=self.left_lung_surface_mm[:, 1],
                    z=self.left_lung_surface_mm[:, 0],
                ))
            else:
                frame_data.append(go.Scatter3d(x=[], y=[], z=[]))
                
            if self.right_lung_surface_mm is not None:
                frame_data.append(go.Scatter3d(
                    x=self.right_lung_surface_mm[:, 2],
                    y=self.right_lung_surface_mm[:, 1],
                    z=self.right_lung_surface_mm[:, 0],
                ))
            else:
                frame_data.append(go.Scatter3d(x=[], y=[], z=[]))
            
            for state in cell_order:
                cell_data = state_data['cells'][state]
                frame_data.append(go.Scatter3d(
                    x=cell_data['x'], y=cell_data['y'], z=cell_data['z'],
                ))
            
            frames.append(go.Frame(
                data=frame_data,
                name=str(state_data['iteration']),
                traces=[0, 1, 2, 3, 4, 5]
            ))
        
        fig = go.Figure(data=traces, frames=frames)
        
        updatemenus = [
            dict(
                type="buttons",
                showactive=False,
                y=1.02,
                x=0.0,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[None, {
                            "frame": {"duration": 300, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 100}
                        }]
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[[None], {
                            "frame": {"duration": 0, "redraw": False},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }]
                    ),
                    dict(
                        label="⏮ Reset",
                        method="animate",
                        args=[[str(self.simulation_states[0]['iteration'])], {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0}
                        }]
                    )
                ]
            ),
        ]
        
        sliders = [
            dict(
                active=0,
                yanchor="top",
                xanchor="left",
                currentvalue=dict(
                    font=dict(size=14),
                    prefix="Iteration: ",
                    visible=True,
                    xanchor="center"
                ),
                transition=dict(duration=100),
                pad=dict(b=10, t=50),
                len=0.9,
                x=0.05,
                y=0,
                steps=[
                    dict(
                        args=[[str(state['iteration'])],
                              dict(frame=dict(duration=0, redraw=True),
                                   mode="immediate",
                                   transition=dict(duration=0))],
                        label=str(state['iteration']),
                        method="animate"
                    )
                    for state in self.simulation_states
                ]
            )
        ]
        
        fig.update_layout(
            title=dict(
                text=f"<b>🫁 {patient_id} - {risk_label}</b><br>" +
                     f"<sup>α={p_alpha:.4f} | β={p_necrosis:.4f}</sup>",
                x=0.5,
                font=dict(size=18)
            ),
            scene=dict(
                xaxis=dict(title='X (mm)', backgroundcolor='rgb(240,245,250)',
                          gridcolor='white', showbackground=True),
                yaxis=dict(title='Y (mm)', backgroundcolor='rgb(240,245,250)',
                          gridcolor='white', showbackground=True),
                zaxis=dict(title='Z (mm)', backgroundcolor='rgb(240,245,250)',
                          gridcolor='white', showbackground=True),
                aspectmode='data',
                camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
            ),
            width=1100,
            height=850,
            updatemenus=updatemenus,
            sliders=sliders,
            legend=dict(
                yanchor="top", y=0.95,
                xanchor="right", x=0.99,
                bgcolor="rgba(255,255,255,0.9)",
                bordercolor="gray", borderwidth=1
            ),
            margin=dict(l=0, r=0, t=100, b=120)
        )
        
        html_content = fig.to_html(include_plotlyjs=True, full_html=True)
        
        custom_controls = """
<style>
    .control-panel {
        position: fixed;
        top: 10px;
        right: 10px;
        background: rgba(255,255,255,0.95);
        padding: 15px 20px;
        border-radius: 12px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        z-index: 1000;
        font-family: 'Segoe UI', Arial, sans-serif;
        min-width: 200px;
    }
    .control-panel h3 {
        margin: 0 0 12px 0;
        font-size: 14px;
        color: #333;
        border-bottom: 1px solid #eee;
        padding-bottom: 8px;
    }
    .slider-row {
        display: flex;
        align-items: center;
        margin: 10px 0;
    }
    .slider-row label {
        flex: 1;
        font-size: 13px;
        color: #555;
    }
    .slider-row input[type="range"] {
        width: 100px;
        cursor: pointer;
    }
    .slider-value {
        width: 45px;
        text-align: right;
        font-family: 'Consolas', monospace;
        font-size: 12px;
        color: #666;
    }
    .legend-section {
        margin-top: 15px;
        padding-top: 12px;
        border-top: 1px solid #eee;
    }
    .legend-section h4 {
        margin: 0 0 8px 0;
        font-size: 12px;
        color: #888;
    }
    .legend-item {
        display: flex;
        align-items: center;
        margin: 4px 0;
        font-size: 11px;
    }
    .legend-dot {
        width: 10px;
        height: 10px;
        border-radius: 50%;
        margin-right: 8px;
        flex-shrink: 0;
    }
</style>

<div class="control-panel">
    <h3>🎛️ Display Controls</h3>
    
    <div class="slider-row">
        <label>🫁 Lung Opacity</label>
        <input type="range" id="lungOpacity" min="0" max="50" value="15" 
               oninput="updateLungOpacity(this.value)">
        <span class="slider-value" id="opacityValue">0.15</span>
    </div>
    
    <div class="legend-section">
        <h4>LEGEND</h4>
        <div class="legend-item">
            <div class="legend-dot" style="background: dodgerblue;"></div>
            <span>Left Lung</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: limegreen;"></div>
            <span>Right Lung</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: darkblue;"></div>
            <span>Tumor Core</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: red;"></div>
            <span>Proliferating</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: purple;"></div>
            <span>Malignant</span>
        </div>
        <div class="legend-item">
            <div class="legend-dot" style="background: dimgray;"></div>
            <span>Necrotic</span>
        </div>
    </div>
</div>

<script>
function updateLungOpacity(value) {
    var opacity = value / 100;
    document.getElementById('opacityValue').textContent = opacity.toFixed(2);
    
    var plotDiv = document.getElementsByClassName('plotly-graph-div')[0];
    if (plotDiv) {
        Plotly.restyle(plotDiv, {'marker.opacity': opacity}, [0, 1]);
    }
}

document.addEventListener('DOMContentLoaded', function() {
    setTimeout(function() {
        updateLungOpacity(15);
    }, 500);
});
</script>
"""
        
        html_content = html_content.replace('</body>', custom_controls + '</body>')
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print(f"   ✅ Đã lưu: {output_path}")

    def run(self, lung_mask, tumor_mask, ct_volume, spacing, origin,
            p_alpha, p_necrosis, patient_id, risk_label) -> np.ndarray:
        """Chạy mô phỏng và tạo visualization."""
        print(f"\n🔄 Mô phỏng 3D cho {patient_id} ({risk_label})...")
        print(f"   📊 Tham số: α={p_alpha:.4f}, β={p_necrosis:.4f}")
        
        self.simulation_states = []
        
        self.transform = RealSpaceTransform(spacing, origin, lung_mask.shape, self.grid_size)

        lung_grid = self.resample(lung_mask, lung_mask.shape)
        tumor_grid = self.resample(tumor_mask, tumor_mask.shape)
        tumor_grid = tumor_grid & lung_grid

        self.prepare_lung_surfaces(lung_grid)

        grid = np.zeros_like(lung_grid, dtype=np.int8)
        grid[lung_grid] = 1
        grid[tumor_grid] = 2

        scale = np.array([self.grid_size] * 3) / np.array(ct_volume.shape)
        ct_small = zoom(ct_volume, scale, order=1)
        microenv = np.clip((ct_small + 1000) / 1400, 0, 1)

        self.capture_state(grid, 0)

        for it in range(self.cfg['num_iterations']):
            tumor_cells = (grid == 2) | (grid == 4) | (grid == 6)
            
            dilated = binary_dilation(tumor_cells)
            boundary = dilated & (grid == 1) & lung_grid
            candidates = np.argwhere(boundary)

            if len(candidates) > 0:
                n_grow = int(len(candidates) * p_alpha)
                if n_grow > 0:
                    weights = microenv[tuple(candidates.T)]
                    weights /= (weights.sum() + 1e-9)
                    indices = np.random.choice(len(candidates), 
                                              size=min(n_grow, len(candidates)),
                                              p=weights, replace=False)
                    for idx in indices:
                        grid[tuple(candidates[idx])] = 4

            mask_4 = (grid == 4)
            grid[mask_4 & (np.random.rand(*grid.shape) < 0.2)] = 2

            mask_2 = (grid == 2)
            grid[mask_2 & (np.random.rand(*grid.shape) < self.cfg['mutation_chance'])] = 6

            if it > self.cfg.get('necrosis_delay', 15):
                eroded = binary_erosion(tumor_cells, iterations=2)
                grid[eroded & (grid == 2) & (np.random.rand(*grid.shape) < p_necrosis)] = 3

            grid[~lung_grid] = 0

            if (it + 1) % self.cfg['capture_interval'] == 0:
                print(f"   📸 Iteration {it+1}/{self.cfg['num_iterations']}")
                self.capture_state(grid, it + 1)

        html_path = OUTPUT_DIR / f"{patient_id}_{risk_label}_digital_twin.html"
        self.create_interactive_html(patient_id, risk_label, p_alpha, p_necrosis, html_path)
        
        return grid


# ============================================================================
# 4. MAIN PIPELINE
# ============================================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "=" * 80)
    print("🚀 RADIOMICS-DRIVEN DIGITAL TWIN + INTERACTIVE VISUALIZATION")
    print(f"   Mode: {'DEBUG (5 patients)' if DEBUG_MODE else 'FULL RUN'}")
    print("=" * 80)
    
    # ========================================================================
    # PHASE 1: Load Clinical Data
    # ========================================================================
    if not CLINICAL_FILE.exists():
        print(f"❌ Error: Không tìm thấy file: {CLINICAL_FILE}")
        return

    df_clin = pd.read_csv(CLINICAL_FILE)
    df_clin.columns = [c.strip() for c in df_clin.columns]

    col_map = {
        'PatientID': 'PatientID', 'age': 'age',
        'Survival.time': 'time', 'deadstatus.event': 'event', 'gender': 'gender'
    }
    df_clin = df_clin.rename(columns=col_map)
    df_clin['PatientID'] = df_clin['PatientID'].astype(str)

    if 'gender' in df_clin.columns:
        df_clin['gender'] = df_clin['gender'].astype(str).str.lower().map(
            {'male': 1, 'female': 0}).fillna(1)
    else:
        df_clin['gender'] = 1

    df_clin['time'] = pd.to_numeric(df_clin['time'], errors='coerce')
    df_clin['event'] = pd.to_numeric(df_clin['event'], errors='coerce')
    df_clin = df_clin.dropna(subset=['time', 'event', 'age'])

    patient_list = df_clin['PatientID'].unique().tolist()
    if DEBUG_MODE:
        patient_list = patient_list[:5]

    # ========================================================================
    # PHASE 2: Batch Simulation
    # ========================================================================
    print(f"\n📊 PHASE 2: Batch Simulation ({len(patient_list)} patients)...")
    
    results = Parallel(n_jobs=N_JOBS)(
        delayed(process_wrapper)(pid) for pid in tqdm(patient_list, desc="Simulating")
    )

    valid_results = [r for r in results if r is not None]
    print(f"\n✅ Complete: {len(valid_results)} valid patients")

    if len(valid_results) <= 10:
        print("❌ Không đủ dữ liệu để phân tích.")
        return

    # ========================================================================
    # Print Biological Parameters for All Patients
    # ========================================================================
    print("\n" + "=" * 80)
    print("📋 THAM SỐ SINH HỌC CHO TẤT CẢ BỆNH NHÂN")
    print("=" * 80)
    
    params_df = pd.DataFrame(valid_results)
    params_df = params_df.sort_values('Bio_Alpha', ascending=False)
    
    alpha_unique = params_df['Bio_Alpha'].nunique()
    alpha_std = params_df['Bio_Alpha'].std()
    
    print(f"\n✅ Validation Check:")
    print(f"   Unique α values: {alpha_unique}/{len(params_df)}")
    print(f"   α std deviation: {alpha_std:.6f}")
    
    if alpha_unique == 1:
        print("   ⚠️ WARNING: All α values are identical! Formula may need adjustment.")
    elif alpha_std < 0.01:
        print("   ⚠️ WARNING: Very low α variation! Formula may need adjustment.")
    else:
        print("   ✅ Good diversity in α values!")
    
    print(f"\n{'No.':<5} {'PatientID':<20} {'α (Proliferation)':<20} {'β (Necrosis)':<20} {'Entropy':<12} {'Sphericity':<12}")
    print("=" * 89)
    
    for idx, row in enumerate(params_df.itertuples(), 1):
        print(f"{idx:<5} {row.PatientID:<20} {row.Bio_Alpha:<20.6f} {row.Bio_Necrosis:<20.6f} {row.Radiomics_Entropy:<12.4f} {row.Radiomics_Sphericity:<12.4f}")
    
    print("\n" + "=" * 80)
    print("📊 THỐNG KÊ TỔNG QUAN:")
    print("=" * 80)
    print(f"   α (Proliferation Rate):")
    print(f"      Mean:   {params_df['Bio_Alpha'].mean():.6f}")
    print(f"      Median: {params_df['Bio_Alpha'].median():.6f}")
    print(f"      Std:    {params_df['Bio_Alpha'].std():.6f}")
    print(f"      Range:  [{params_df['Bio_Alpha'].min():.6f}, {params_df['Bio_Alpha'].max():.6f}]")
    print(f"\n   β (Necrosis Rate):")
    print(f"      Mean:   {params_df['Bio_Necrosis'].mean():.6f}")
    print(f"      Median: {params_df['Bio_Necrosis'].median():.6f}")
    print(f"      Std:    {params_df['Bio_Necrosis'].std():.6f}")
    print(f"      Range:  [{params_df['Bio_Necrosis'].min():.6f}, {params_df['Bio_Necrosis'].max():.6f}]")
    
    params_output = OUTPUT_DIR / "all_patients_biological_parameters.csv"
    params_df_save = params_df[['PatientID', 'Bio_Alpha', 'Bio_Necrosis', 
                                 'Radiomics_Entropy', 'Radiomics_Sphericity',
                                 'Sim_Growth_Rate', 'Sim_Necrosis_Ratio']]
    params_df_save.to_csv(params_output, index=False)
    print(f"\n✅ Đã lưu tham số vào: {params_output}")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax1 = axes[0, 0]
    ax1.hist(params_df['Bio_Alpha'], bins=30, color='red', alpha=0.7, edgecolor='black')
    ax1.axvline(params_df['Bio_Alpha'].mean(), color='darkred', linestyle='--', 
                linewidth=2, label=f'Mean: {params_df["Bio_Alpha"].mean():.4f}')
    ax1.axvline(params_df['Bio_Alpha'].median(), color='orange', linestyle='-.', 
                linewidth=2, label=f'Median: {params_df["Bio_Alpha"].median():.4f}')
    ax1.set_xlabel('α (Proliferation Rate)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax1.set_title('Distribution of Proliferation Rate (α)', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[0, 1]
    ax2.hist(params_df['Bio_Necrosis'], bins=30, color='purple', alpha=0.7, edgecolor='black')
    ax2.axvline(params_df['Bio_Necrosis'].mean(), color='darkviolet', linestyle='--', 
                linewidth=2, label=f'Mean: {params_df["Bio_Necrosis"].mean():.4f}')
    ax2.axvline(params_df['Bio_Necrosis'].median(), color='magenta', linestyle='-.', 
                linewidth=2, label=f'Median: {params_df["Bio_Necrosis"].median():.4f}')
    ax2.set_xlabel('β (Necrosis Rate)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of Necrosis Rate (β)', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    ax3 = axes[1, 0]
    scatter = ax3.scatter(params_df['Bio_Alpha'], params_df['Bio_Necrosis'], 
                         c=params_df['Radiomics_Entropy'], cmap='viridis', 
                         s=100, alpha=0.6, edgecolor='black')
    ax3.set_xlabel('α (Proliferation Rate)', fontsize=12, fontweight='bold')
    ax3.set_ylabel('β (Necrosis Rate)', fontsize=12, fontweight='bold')
    ax3.set_title('α vs β Relationship (colored by Entropy)', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax3)
    cbar.set_label('Entropy', fontsize=10, fontweight='bold')
    
    ax4 = axes[1, 1]
    top10 = params_df.head(10)
    y_pos = np.arange(len(top10))
    ax4.barh(y_pos, top10['Bio_Alpha'].values, color='crimson', alpha=0.7, edgecolor='black')
    ax4.set_yticks(y_pos)
    ax4.set_yticklabels([pid[:15] + '...' if len(pid) > 15 else pid 
                         for pid in top10['PatientID'].values], fontsize=9)
    ax4.set_xlabel('α (Proliferation Rate)', fontsize=12, fontweight='bold')
    ax4.set_title('Top 10 Patients by Proliferation Rate', fontsize=13, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='x')
    ax4.invert_yaxis()
    
    plt.tight_layout()
    params_plot = OUTPUT_DIR / "biological_parameters_distribution.png"
    plt.savefig(params_plot, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Đã lưu biểu đồ phân bố: {params_plot}")
    print("=" * 80 + "\n")

    # ========================================================================
    # PHASE 3: Cox Survival Analysis + Stratified K-Fold CV
    # ========================================================================
    print("\n📊 PHASE 3: Cox Survival Analysis + Cross-Validation...")

    sim_df = pd.DataFrame(valid_results).set_index('PatientID')
    df_clin_indexed = df_clin.set_index('PatientID')
    final_df = df_clin_indexed.join(sim_df, how='inner')
    final_df['time_years'] = final_df['time'] / 365.25

    feature_list = ['Sim_Growth_Rate', 'Sim_Necrosis_Ratio', 'age', 'gender']
    final_df = final_df.dropna(subset=feature_list + ['time_years', 'event'])

    scaler = StandardScaler()
    df_scaled = final_df.copy()
    df_scaled[feature_list] = scaler.fit_transform(final_df[feature_list])

    cph = CoxPHFitter()
    cph.fit(df_scaled[feature_list + ['time_years', 'event']],
            duration_col='time_years', event_col='event')

    print(f"\n🔹 Full Model C-Index: {cph.concordance_index_:.4f}")
    print(f"\n📋 Cox Model Summary:")
    print(cph.summary[['coef', 'exp(coef)', 'p']])

    # Stratified K-Fold Cross-Validation
    print("\n" + "=" * 60)
    print("📊 STRATIFIED K-FOLD CROSS-VALIDATION")
    print("=" * 60)
    
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    stratify_labels = final_df['event'].values.astype(int)
    
    cv_scores = []
    fold_results = []
    
    print(f"\nRunning {n_splits}-Fold Cross-Validation...")
    print(f"Total samples: {len(df_scaled)} | Events: {int(final_df['event'].sum())}\n")
    
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df_scaled, stratify_labels), 1):
        train_data = df_scaled.iloc[train_idx]
        test_data = df_scaled.iloc[test_idx]
        
        cph_fold = CoxPHFitter()
        cph_fold.fit(train_data[feature_list + ['time_years', 'event']],
                     duration_col='time_years', event_col='event')
        
        c_index_test = concordance_index(
            test_data['time_years'],
            -cph_fold.predict_partial_hazard(test_data[feature_list]),
            test_data['event']
        )
        
        cv_scores.append(c_index_test)
        fold_results.append({
            'Fold': fold_idx,
            'Train_Size': len(train_idx),
            'Test_Size': len(test_idx),
            'Train_Events': int(train_data['event'].sum()),
            'Test_Events': int(test_data['event'].sum()),
            'C_Index': c_index_test
        })
        
        print(f"   Fold {fold_idx}/{n_splits}: C-Index = {c_index_test:.4f} " +
              f"(Train: {len(train_idx)}, Test: {len(test_idx)}, " +
              f"Test Events: {int(test_data['event'].sum())})")
    
    cv_mean = np.mean(cv_scores)
    cv_std = np.std(cv_scores)
    cv_min = np.min(cv_scores)
    cv_max = np.max(cv_scores)
    
    print(f"\n{'='*60}")
    print(f"📊 CROSS-VALIDATION SUMMARY:")
    print(f"{'='*60}")
    print(f"   Mean C-Index:     {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"   Range:            [{cv_min:.4f}, {cv_max:.4f}]")
    print(f"   Full Model:       {cph.concordance_index_:.4f}")
    print(f"   Performance:      {'✅ Stable' if cv_std < 0.05 else '⚠️ Variable'}")
    print(f"{'='*60}\n")
    
    cv_results_df = pd.DataFrame(fold_results)
    cv_path = OUTPUT_DIR / "cross_validation_results.csv"
    cv_results_df.to_csv(cv_path, index=False)
    print(f"✅ Saved CV results: {cv_path}")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.bar(range(1, n_splits + 1), cv_scores, color='steelblue', alpha=0.7, 
            edgecolor='black', linewidth=1.5)
    ax1.axhline(y=cv_mean, color='red', linestyle='--', linewidth=2, 
                label=f'CV Mean: {cv_mean:.4f}')
    ax1.axhline(y=cph.concordance_index_, color='green', linestyle='-.', linewidth=2, 
                label=f'Full Model: {cph.concordance_index_:.4f}')
    ax1.set_xlabel('Fold', fontsize=12, fontweight='bold')
    ax1.set_ylabel('C-Index', fontsize=12, fontweight='bold')
    ax1.set_title('Cross-Validation C-Index by Fold', fontsize=13, fontweight='bold')
    ax1.set_ylim([max(0, cv_min - 0.05), min(1, cv_max + 0.05)])
    ax1.set_xticks(range(1, n_splits + 1))
    ax1.legend(fontsize=10, loc='lower right')
    ax1.grid(True, alpha=0.3, linestyle='--')
    
    for i, v in enumerate(cv_scores, 1):
        ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', 
                fontsize=9, fontweight='bold')
    
    ax2.boxplot([cv_scores], labels=['CV C-Index'], 
                patch_artist=True, 
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2),
                widths=0.5)
    ax2.scatter([1]*len(cv_scores), cv_scores, color='darkblue', s=100, 
                alpha=0.6, zorder=3, label='Fold Results')
    ax2.axhline(y=cph.concordance_index_, color='green', linestyle='-.', 
                linewidth=2, label=f'Full Model: {cph.concordance_index_:.4f}')
    ax2.set_ylabel('C-Index', fontsize=12, fontweight='bold')
    ax2.set_title('Distribution of CV C-Index Scores', fontsize=13, fontweight='bold')
    ax2.set_ylim([max(0, cv_min - 0.05), min(1, cv_max + 0.05)])
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    stats_text = f'Mean: {cv_mean:.4f}\nStd: {cv_std:.4f}\nMin: {cv_min:.4f}\nMax: {cv_max:.4f}'
    ax2.text(0.98, 0.02, stats_text, transform=ax2.transAxes,
            fontsize=9, verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    cv_plot_path = OUTPUT_DIR / "cross_validation_plot.png"
    plt.savefig(cv_plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved CV plot: {cv_plot_path}\n")

    # ========================================================================
    # PHASE 4: Identify Representative Patients (2 High + 2 Low)
    # ========================================================================
    print("\n" + "=" * 60)
    print("🔬 PHASE 4: CASE STUDY - 2 HIGH RISK + 2 LOW RISK")
    print("=" * 60)

    risk_scores = cph.predict_partial_hazard(df_scaled)
    final_df['Risk_Score'] = risk_scores

    # Get top 2 highest risk and bottom 2 lowest risk
    risk_sorted = final_df.sort_values('Risk_Score', ascending=False)
    # Chia làm 2 nhóm: high risk (trên median) và low risk (dưới median)
    median_risk = risk_sorted['Risk_Score'].median()
    high_risk_pool = risk_sorted[risk_sorted['Risk_Score'] > median_risk]
    low_risk_pool = risk_sorted[risk_sorted['Risk_Score'] <= median_risk]

    # Random sample 5 từ mỗi nhóm
    top_2_high = high_risk_pool.sample(n=min(5, len(high_risk_pool)), random_state=42)
    top_2_low = low_risk_pool.sample(n=min(5, len(low_risk_pool)), random_state=42)

    def print_profile(row, label, pid):
        print(f"\n📌 {label} (Patient: {pid})")
        print(f"   Survival: {row['time_years']:.2f} yrs | " +
              f"Event: {'Dead' if row['event'] == 1 else 'Censored'}")
        print(f"   Age: {row['age']:.0f} | Gender: {'M' if row['gender'] == 1 else 'F'}")
        print(f"   Risk Score: {row['Risk_Score']:.4f}")
        print(f"   α (prolif): {row['Bio_Alpha']:.4f} | β (necro): {row['Bio_Necrosis']:.4f}")

    print("\n🔴 HIGH RISK PATIENTS:")
    for i, (pid, row) in enumerate(top_2_high.iterrows(), 1):
        print_profile(row, f"HIGH RISK #{i}", pid)

    print("\n🟢 LOW RISK PATIENTS:")
    for i, (pid, row) in enumerate(top_2_low.iterrows(), 1):
        print_profile(row, f"LOW RISK #{i}", pid)

    # ========================================================================
    # PHASE 5: Interactive 3D Visualization (4 patients)
    # ========================================================================
    print("\n" + "=" * 60)
    print("🎨 PHASE 5: INTERACTIVE 3D VISUALIZATION (4 patients)")
    print("=" * 60)

    visualization_patients = []
    for i, (pid, row) in enumerate(top_2_high.iterrows(), 1):
        visualization_patients.append((pid, row, f"HIGH_RISK_{i}"))
    for i, (pid, row) in enumerate(top_2_low.iterrows(), 1):
        visualization_patients.append((pid, row, f"LOW_RISK_{i}"))

    for pid, row, label in visualization_patients:
        print(f"\n{'='*40}")
        
        data = load_patient_data_smart(pid, LUNG1_ROOT)
        if data is None:
            print(f"   ⚠️ Không load được data cho {pid}")
            continue

        vol, tumor, lung, spacing, origin = data

        visualizer = MLPA3DVisualizer(MLPA_CONFIG_VIZ)
        visualizer.run(
            lung_mask=lung,
            tumor_mask=tumor,
            ct_volume=vol,
            spacing=spacing,
            origin=origin,
            p_alpha=row['Bio_Alpha'],
            p_necrosis=row['Bio_Necrosis'],
            patient_id=pid,
            risk_label=label
        )

    # ========================================================================
    # PHASE 6: Kaplan-Meier Plot
    # ========================================================================
    print("\n📊 PHASE 6: Kaplan-Meier Survival Curves...")

    plt.figure(figsize=(10, 6))
    
    med_score = risk_scores.median()
    high_group = final_df[risk_scores > med_score]
    low_group = final_df[risk_scores <= med_score]

    kmf = KaplanMeierFitter()
    
    kmf.fit(high_group['time_years'], high_group['event'], label='High Risk')
    kmf.plot_survival_function(color='red', linewidth=2)
    
    kmf.fit(low_group['time_years'], low_group['event'], label='Low Risk')
    kmf.plot_survival_function(color='blue', linewidth=2)

    p_val = logrank_test(
        high_group['time_years'], low_group['time_years'],
        event_observed_A=high_group['event'], event_observed_B=low_group['event']
    ).p_value

    plt.title(f"Digital Twin Prognosis (Log-Rank p={p_val:.5f})", fontsize=14)
    plt.xlabel("Years", fontsize=12)
    plt.ylabel("Survival Probability", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=11)

    km_path = OUTPUT_DIR / "kaplan_meier_survival.png"
    plt.savefig(km_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Saved: {km_path}")

    # ========================================================================
    # Summary
    # ========================================================================
    print("\n" + "=" * 60)
    print("✅ HOÀN THÀNH!")
    print("=" * 60)
    print(f"\n📁 Output Directory: {OUTPUT_DIR}/")
    print(f"\n📊 Analysis Results:")
    print(f"   • all_patients_biological_parameters.csv")
    print(f"   • biological_parameters_distribution.png")
    print(f"   • cross_validation_results.csv")
    print(f"   • cross_validation_plot.png")
    print(f"   • kaplan_meier_survival.png")
    print(f"\n🎨 3D Visualizations (4 files):")
    for pid, _, label in visualization_patients:
        print(f"   • {pid}_{label}_digital_twin.html")
    print(f"\n💡 Interactive Features:")
    print(f"   - ▶ Play / ⏸ Pause: Control animation")
    print(f"   - 🎚️ Lung Opacity: Adjust transparency")
    print(f"   - 🖱️ Mouse: Rotate and zoom 3D model")
    print(f"\n📈 Model Performance:")
    print(f"   - Full Model C-Index: {cph.concordance_index_:.4f}")
    print(f"   - CV Mean C-Index: {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"   - Model Stability: {'✅ Good' if cv_std < 0.05 else '⚠️ Check results'}")
    print(f"\n🧬 Biological Parameters:")
    print(f"   - Total Patients: {len(valid_results)}")
    print(f"   - α Range: [{params_df['Bio_Alpha'].min():.4f}, {params_df['Bio_Alpha'].max():.4f}]")
    print(f"   - β Range: [{params_df['Bio_Necrosis'].min():.4f}, {params_df['Bio_Necrosis'].max():.4f}]")
    print(f"\n🔬 Case Study: 4 Representative Patients")
    print(f"   - 2 High Risk patients visualized")
    print(f"   - 2 Low Risk patients visualized")


if __name__ == "__main__":
    main()


🚀 RADIOMICS-DRIVEN DIGITAL TWIN + INTERACTIVE VISUALIZATION
   Mode: FULL RUN

📊 PHASE 2: Batch Simulation (400 patients)...


Simulating: 100%|██████████| 400/400 [04:42<00:00,  1.42it/s]



✅ Complete: 390 valid patients

📋 THAM SỐ SINH HỌC CHO TẤT CẢ BỆNH NHÂN

✅ Validation Check:
   Unique α values: 382/390
   α std deviation: 0.010562
   ✅ Good diversity in α values!

No.   PatientID            α (Proliferation)    β (Necrosis)         Entropy      Sphericity  
1     LUNG1-386            0.060000             0.045167             0.8173       0.5604      
2     LUNG1-296            0.054764             0.052302             0.7372       0.4712      
3     LUNG1-282            0.054372             0.040363             0.7325       0.6205      
4     LUNG1-250            0.052989             0.034350             0.7159       0.6956      
5     LUNG1-167            0.052403             0.054249             0.7088       0.4469      
6     LUNG1-086            0.052251             0.043344             0.7070       0.5832      
7     LUNG1-415            0.051541             0.037787             0.6985       0.6527      
8     LUNG1-094            0.051299             0.03442

✅ Đã lưu biểu đồ phân bố: digital_twin_output/biological_parameters_distribution.png


📊 PHASE 3: Cox Survival Analysis + Cross-Validation...

🔹 Full Model C-Index: 0.5866

📋 Cox Model Summary:
                        coef  exp(coef)         p
covariate                                        
Sim_Growth_Rate     0.038439   1.039188  0.482784
Sim_Necrosis_Ratio  0.240100   1.271376  0.000016
age                 0.195014   1.215328  0.001550
gender              0.080579   1.083914  0.158031

📊 STRATIFIED K-FOLD CROSS-VALIDATION

Running 5-Fold Cross-Validation...
Total samples: 390 | Events: 345

   Fold 1/5: C-Index = 0.5881 (Train: 312, Test: 78, Test Events: 69)
   Fold 2/5: C-Index = 0.5391 (Train: 312, Test: 78, Test Events: 69)
   Fold 3/5: C-Index = 0.5268 (Train: 312, Test: 78, Test Events: 69)
   Fold 4/5: C-Index = 0.6346 (Train: 312, Test: 78, Test Events: 69)
   Fold 5/5: C-Index = 0.6161 (Train: 312, Test: 78, Test Events: 69)

📊 CROSS-VALIDATION SUMMARY:
   Mean C-Index:   

   📸 Iteration 63/120
   📸 Iteration 66/120
   📸 Iteration 69/120
   📸 Iteration 72/120
   📸 Iteration 75/120
   📸 Iteration 78/120
   📸 Iteration 81/120
   📸 Iteration 84/120
   📸 Iteration 87/120
   📸 Iteration 90/120
   📸 Iteration 93/120
   📸 Iteration 96/120
   📸 Iteration 99/120
   📸 Iteration 102/120
   📸 Iteration 105/120
   📸 Iteration 108/120
   📸 Iteration 111/120
   📸 Iteration 114/120
   📸 Iteration 117/120
   📸 Iteration 120/120
   🌐 Đang tạo HTML interactive...
   ✅ Đã lưu: digital_twin_output/LUNG1-015_LOW_RISK_2_digital_twin.html


🔄 Mô phỏng 3D cho LUNG1-183 (LOW_RISK_3)...
   📊 Tham số: α=0.0365, β=0.0599
   🔨 Đang tách và trích xuất bề mặt 2 lá phổi...
      - Left Lung: 1 voxels → 1 points
      - Right Lung: 8343 voxels → 1669 points
   📸 Iteration 3/120
   📸 Iteration 6/120
   📸 Iteration 9/120
   📸 Iteration 12/120
   📸 Iteration 15/120
   📸 Iteration 18/120
   📸 Iteration 21/120
   📸 Iteration 24/120
   📸 Iteration 27/120
   📸 Iteration 30/120
   📸 Iteration 33